# Fire Detection — YOLOv11 Training
**Author:** Alaaeddine Bouchamla — ENISO M1 Telecommunications

**Before running:** Runtime → Change runtime type → **T4 GPU**

Steps: download D-Fire dataset → train YOLOv11n + YOLOv11x → evaluate → export ONNX

## 0. Install dependencies

In [ ]:
!pip install ultralytics kaggle onnxruntime -q
import ultralytics
ultralytics.checks()

## 1. Kaggle credentials

1. Go to **https://www.kaggle.com/settings**
2. Scroll to **API** section → click **Create New Token**
3. It downloads `kaggle.json` — open it and copy the values below

The file looks like: `{"username":"yourname","key":"abc123..."}`

In [ ]:
import os, json

# ── Paste YOUR values here ──────────────────────────────────
KAGGLE_USERNAME = "boshamla4"          # your Kaggle username
KAGGLE_KEY      = "paste_your_key_here" # the long hex string from kaggle.json
# ────────────────────────────────────────────────────────────

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials saved.')

## 2. Download D-Fire dataset

In [ ]:
!kaggle datasets download -d sayedgamal99/smoke-fire-detection-yolo -p /content/
!unzip -q /content/smoke-fire-detection-yolo.zip -d /content/d-fire

# Show structure
for root, dirs, files_list in os.walk('/content/d-fire'):
    level = root.replace('/content/d-fire', '').count(os.sep)
    if level > 2:
        continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files_list)} files)')

## 3. Dataset config

In [ ]:
import yaml

base = '/content/d-fire'
splits = {}
for split in ['train', 'val', 'test', 'valid']:
    p = os.path.join(base, split, 'images')
    if os.path.exists(p):
        key = 'val' if split == 'valid' else split
        splits[key] = p

dataset_config = {
    'path': base,
    'train': splits.get('train', 'train/images'),
    'val':   splits.get('val',   'val/images'),
    'test':  splits.get('test',  'test/images'),
    'nc': 2,
    'names': {0: 'fire', 1: 'smoke'}
}

config_path = '/content/fire-dataset.yaml'
with open(config_path, 'w') as f:
    yaml.dump(dataset_config, f)

print(yaml.dump(dataset_config))

## 4. Train YOLOv11n — edge model (~30–40 min on T4)

In [ ]:
from ultralytics import YOLO

model_n = YOLO('yolo11n.pt')
results_n = model_n.train(
    data=config_path,
    epochs=100,
    batch=16,
    imgsz=640,
    device=0,
    name='fire-yolo11n',
    optimizer='AdamW',
    lr0=0.01,
    cos_lr=True,
    warmup_epochs=3,
    mosaic=1.0,
    degrees=10.0,
    plots=True,
    save=True,
)

WEIGHTS_N = f'{results_n.save_dir}/weights/best.pt'
print(f'YOLOv11n done → {WEIGHTS_N}')

## 5. Train YOLOv11x — cloud baseline (~90–120 min on T4)

In [ ]:
model_x = YOLO('yolo11x.pt')
results_x = model_x.train(
    data=config_path,
    epochs=100,
    batch=8,
    imgsz=640,
    device=0,
    name='fire-yolo11x',
    optimizer='AdamW',
    lr0=0.01,
    cos_lr=True,
    warmup_epochs=3,
    mosaic=1.0,
    plots=True,
    save=True,
)

WEIGHTS_X = f'{results_x.save_dir}/weights/best.pt'
print(f'YOLOv11x done → {WEIGHTS_X}')

## 6. Evaluate — paper metrics

In [ ]:
import json

def evaluate(weights, label):
    m = YOLO(weights).val(data=config_path, split='test', plots=True)
    return {
        'model': label,
        'precision':  round(float(m.box.mp),    4),
        'recall':     round(float(m.box.mr),    4),
        'mAP50':      round(float(m.box.map50), 4),
        'mAP50_95':   round(float(m.box.map),   4),
        'per_class': {
            name: {'precision': round(float(p),4), 'recall': round(float(r),4), 'mAP50': round(float(a),4)}
            for name, p, r, a in zip(m.names.values(), m.box.p, m.box.r, m.box.ap50)
        }
    }

metrics_n = evaluate(WEIGHTS_N, 'YOLOv11n (edge, FP32)')
metrics_x = evaluate(WEIGHTS_X, 'YOLOv11x (cloud)')
all_metrics = [metrics_n, metrics_x]

print('\n══ PAPER METRICS — copy these into paper.md ══')
print(f'{"Model":<30} {"P":>6} {"R":>6} {"mAP50":>7} {"mAP50-95":>10}')
print('─' * 65)
for m in all_metrics:
    print(f"{m['model']:<30} {m['precision']:>6.4f} {m['recall']:>6.4f} {m['mAP50']:>7.4f} {m['mAP50_95']:>10.4f}")
print('\nPer-class mAP@0.5:')
for m in all_metrics:
    print(f"  {m['model']}:")
    for cls, v in m['per_class'].items():
        print(f"    {cls}: mAP50={v['mAP50']:.4f}")

with open('/content/metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

## 7. Export YOLOv11n to ONNX + benchmark CPU latency

In [ ]:
import time
import numpy as np
import onnxruntime as ort

onnx_path = YOLO(WEIGHTS_N).export(format='onnx', imgsz=640, simplify=True)

session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
input_name = session.get_inputs()[0].name

for _ in range(5):  # warm-up
    session.run(None, {input_name: dummy})

times = [(time.perf_counter(), session.run(None, {input_name: dummy}), time.perf_counter())
         for _ in range(50)]
avg_ms = sum((t1 - t0) * 1000 for t0, _, t1 in times) / 50

network_ms = 30    # 4G/LTE estimated
cloud_ms   = 50    # DB write + WebSocket push
render_ms  = 100   # browser
total_ms   = avg_ms + network_ms + cloud_ms + render_ms

print(f'\n══ LATENCY BREAKDOWN — copy into paper.md ══')
print(f'  Edge inference (ONNX CPU):  {avg_ms:.1f} ms')
print(f'  4G/LTE uplink (estimated):  {network_ms} ms')
print(f'  Cloud DB write + push:      {cloud_ms} ms')
print(f'  Dashboard render:           {render_ms} ms')
print(f'  ──────────────────────────────────────────')
print(f'  Total end-to-end:           {total_ms:.0f} ms  ({total_ms/1000:.2f} s)')
print(f'  Target: < 5000 ms  →  {"✓ PASS" if total_ms < 5000 else "✗ FAIL"}')

## 8. Download everything

In [ ]:
import shutil
from google.colab import files

os.makedirs('/content/export', exist_ok=True)
shutil.copy(str(onnx_path),        '/content/export/yolo11n-fire.onnx')
shutil.copy('/content/metrics.json','/content/export/metrics.json')
shutil.copy(WEIGHTS_N,             '/content/export/yolo11n-fire-best.pt')
shutil.copy(WEIGHTS_X,             '/content/export/yolo11x-fire-best.pt')

shutil.make_archive('/content/fire_results', 'zip', '/content/runs')
shutil.make_archive('/content/fire_export',  'zip', '/content/export')

files.download('/content/fire_results.zip')   # training curves, confusion matrix
files.download('/content/fire_export.zip')    # ONNX model + metrics.json